<a href="https://colab.research.google.com/github/BraedynL0530/CanopyWatch/blob/master/deforestation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#network stuff here when i leave collab

In [ ]:
import io
import requests
import rasterio
import numpy as np

#DO NOT RUN LOCALLY MY PC WILL EXPLODE!!!!!!!!!!
fastapi_url = "http://127.0.0.1:8000/api/get-satellite-patch" #wont work in google collab(unless i tunnell) this is for deploy/temp link
print("Fetching data from server...")
response = requests.get(fastapi_url)

if response.status_code == 204:
    print("No large NVDI change")
    exit()
elif response.status_code != 200:
    print(f"Error fetching: {response.status_code}")
    exit()

tiff_bytes = response.content

with rasterio.open(io.BytesIO(tiff_bytes)) as src:
    print("--- Raster Metadata ---")
    print(f"Width x height: {src.width}x{src.height}")
    print(f"Band Count: {src.count}")
    print(f"Coordinate System: {src.crs}")
    print(f"Geotransform Matrix:\n{src.transform}")
    print("-----------------------")

    #Shape: (4, 512, 512)  (Bands, Height, Width)
    full_stack = src.read()

    #slice the first 3 bands out for visual evidence / frontend
    rgb_channels = full_stack[0:3, :, :]  # Shape: (3, 512, 512)


    nir_channel = full_stack[3, :, :]     # Shape: (512, 512)

print("!!unpacked channels!!")

In [ ]:
#model
import torch
import torch.nn as nn

class forestClassifier(nn.Module):
    def __init__(self):
        super(forestClassifier, self).__init__()
        self.encoder = nn.conv2d(4, 64, kernel_size=3,  padding=1)
        self.pool = nn.MaxPool2d(kernal_size=2, stride=2)

        self.bottleneckConv = nn.conv2d(64, 128, kernel_size=3, padding=1)
        self.upsample = nn.conv2d(128,64,kernel_size=2,stride=2)

        self.finalConv = nn.conv2D(4,1,kernal_size=1)
    def forward(self, x):
        x1 = self.encoder(x)
        x2 = self.pool(x1)
        b=self.bottleneckConv(x2)
        u=self.upsample(b)
        out = self.finalConv(u)
        return out



In [ ]:
#training loop

In [ ]:
#porbally more netowrok styuff + infrence with model/
#